In [275]:
import pandas as pd
from statsmodels.tsa.api import VARMAX
from statsmodels.tsa.api import ARIMA
from hmmlearn import hmm
from collections import Counter

#### Goals

- figure out how many unique fruits are represented (22)
- figure out how many weeks are missing data for each fruit
- fill in data with averages or drop as needed

In [276]:
produce_dataset = pd.read_csv("ProductPriceIndex.csv")

In [277]:
print(produce_dataset.columns)

# preliminary cleaning, convert "date" column to datetime
produce_dataset["date"] = pd.to_datetime(produce_dataset["date"])

# convert prices to numeric
price_columns = ['farmprice', 'atlantaretail', 'chicagoretail', 'losangelesretail', 'newyorkretail']
for col in price_columns:   # got help from AI to get rid of the $ and conver to float
    produce_dataset[col] = produce_dataset[col].str.replace("$", "").pipe(pd.to_numeric, errors="coerce")

Index(['productname', 'date', 'farmprice', 'atlantaretail', 'chicagoretail',
       'losangelesretail', 'newyorkretail', 'averagespread'],
      dtype='str')


In [278]:
# now check for nan values
produce_dataset.isna().sum()

productname         0
date                0
farmprice           1
atlantaretail       1
chicagoretail       0
losangelesretail    0
newyorkretail       8
averagespread       0
dtype: int64

In [279]:
unique_fruits = list(set(produce_dataset["productname"]))
print(len(unique_fruits), unique_fruits)

22 ['Avocados', 'Oranges', 'Asparagus', 'Honeydews', 'Carrots', 'Nectarines', 'Strawberries', 'Red Leaf Lettuce', 'Cauliflower', 'Iceberg Lettuce', 'Flame Grapes', 'Peaches', 'Broccoli Crowns', 'Celery', 'Tomatoes', 'Cantaloupe', 'Plums', 'Green Leaf Lettuce', 'Thompson Grapes', 'Romaine Lettuce', 'Broccoli Bunches', 'Potatoes']


In [280]:
# group by fruit
by_fruit = produce_dataset.groupby("productname")["date"]
print(f"Maximum dates for which data is collected: {max(by_fruit.nunique())}")
print(f"Number of fruits with over 800 days recorded: {sum(by_fruit.nunique() > 800)}")

mask = by_fruit.nunique() > 800
print(min(by_fruit.apply(set)[mask]["Broccoli Crowns"]))
print(max(by_fruit.apply(set)[mask]["Broccoli Crowns"]))

Maximum dates for which data is collected: 1011
Number of fruits with over 800 days recorded: 12
1999-10-24 00:00:00


2019-05-19 00:00:00


In [281]:
# use pivot_table to get table with NaN for missing dates
pivot = produce_dataset.pivot_table(index="date", columns="productname", values='newyorkretail')
pivot.isna().sum() / len(produce_dataset.index)

productname
Asparagus             0.048268
Avocados              0.019853
Broccoli Bunches      0.003235
Broccoli Crowns       0.000571
Cantaloupe            0.032982
Carrots               0.000761
Cauliflower           0.000571
Celery                0.000761
Flame Grapes          0.033870
Green Leaf Lettuce    0.000571
Honeydews             0.035583
Iceberg Lettuce       0.000634
Nectarines            0.045731
Oranges               0.011227
Peaches               0.044590
Plums                 0.049727
Potatoes              0.010846
Red Leaf Lettuce      0.000761
Romaine Lettuce       0.000825
Strawberries          0.003235
Thompson Grapes       0.047317
Tomatoes              0.035646
dtype: float64

Since there are less than $5\%$ `NaN` for each product, we will keep all of them.

In [290]:
# print(produce_dataset.pivot_table(index="date", columns="productname", values="newyorkretail").isna().sum())

# fill nan values for each price index with linear interpolation
# cobbled together from AI assistance and my own intuition
new_dataframe = pd.DataFrame()
for col in price_columns:
    filled = produce_dataset.pivot_table(index='date', columns='productname', values=col).interpolate(axis=0).ffill().bfill()
    melted = filled.reset_index().melt(id_vars='date', var_name='productname', value_name=col)
    if col == price_columns[0]:  # only need to set these once
        new_dataframe.index = melted.index
        new_dataframe["productname"] = melted["productname"]
        new_dataframe["date"] = melted["date"]
        
    # set the rest of the columns every time
    new_dataframe[col] = melted[col]

print(new_dataframe.pivot_table(index="date", columns="productname", values="newyorkretail").isna().sum())
# it looks like it worked, there aren't any NaN values anymore

productname
Asparagus             0
Avocados              0
Broccoli Bunches      0
Broccoli Crowns       0
Cantaloupe            0
Carrots               0
Cauliflower           0
Celery                0
Flame Grapes          0
Green Leaf Lettuce    0
Honeydews             0
Iceberg Lettuce       0
Nectarines            0
Oranges               0
Peaches               0
Plums                 0
Potatoes              0
Red Leaf Lettuce      0
Romaine Lettuce       0
Strawberries          0
Thompson Grapes       0
Tomatoes              0
dtype: int64


In [286]:
print(len(produce_dataset["averagespread"]))
print(len(new_dataframe["newyorkretail"]))

15766
22418


In [292]:
# convert cleaned data to csv
new_dataframe.to_csv("cleaned_produce.csv")